# Model Training and Comparison Notebook

This notebook trains multiple models to predict customer churn, evaluates them against overfitting and data leakage, compares their performance metrics, selects the best model, and saves all reports and visualization charts.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, 
    confusion_matrix, classification_report, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Set up path to backend for importing components
sys.path.append(os.path.abspath(os.path.join('..', 'backend')))
from app.core.preprocessing import SubscriptionPreprocessor

print("Setup completed successfully.")

## 1. Load Dataset

We load the main dataset `Subscription Fatigue.csv` from the root directory.

In [ ]:
data_path = "../Subscription Fatigue.csv"
df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
print("\nDataset columns:", list(df.columns))
print("\nMissing values:")
print(df.isnull().sum())

print("\nClass balance:")
print(df['Will_Cancel_Next_3_Months'].value_counts(normalize=True))

## 2. Data Leakage and Suspicious Column Check

Identify columns that are identifiers or look like target indicators. We drop them before training to prevent artificial score inflation.

In [ ]:
target_col = "Will_Cancel_Next_3_Months"

# Scan columns for possible leaks
leaks = []
for col in df.columns:
    if col == target_col:
        continue
    col_lower = col.lower()
    if any(k in col_lower for k in ["cancel", "churn", "will_cancel", "target", "prediction"]):
        leaks.append(col)
    elif any(k in col_lower for k in ["customer_id", "cust_id", "uuid"]):
        leaks.append(col)

print("Columns flagged as potential leaks or IDs:", leaks)
X_raw = df.drop(columns=[target_col] + leaks, errors="ignore")
y = df[target_col].astype(int)
print(f"Features shape after dropping leaks: {X_raw.shape}")

## 3. Stratified Split and Preprocessing

We use the backend's `SubscriptionPreprocessor` to ensure that raw inputs are preprocessed in the exact same format expected by the service.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = SubscriptionPreprocessor()
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print(f"Preprocessed train shape: {X_train.shape}")
print(f"Preprocessed test shape: {X_test.shape}")

## 4. Model Training, Cross-Validation and Evaluation

We train: Logistic Regression, Random Forest, XGBoost, CatBoost, and Gradient Boosting.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42, n_jobs=-1, eval_metric='logloss'),
    "CatBoost": CatBoostClassifier(iterations=200, learning_rate=0.05, depth=6, random_seed=42, verbose=False),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42)
}

results = {}
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, clf in models.items():
    print(f"Training {name}...")
    clf.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = clf.predict(X_train)
    y_test_pred = clf.predict(X_test)
    y_test_prob = clf.predict_proba(X_test)[:, 1]
    
    # Basic metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    accuracy_gap = train_acc - test_acc
    precision = precision_score(y_test, y_test_pred, zero_division=0)
    recall = recall_score(y_test, y_test_pred, zero_division=0)
    f1 = f1_score(y_test, y_test_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_test_prob)
    cm = confusion_matrix(y_test, y_test_pred).tolist()
    
    # Cross-validation
    print(f"Evaluating Cross-Validation for {name}...")
    cv_roc_auc = np.mean(cross_val_score(clf, X_train, y_train, cv=cv_folds, scoring='roc_auc', n_jobs=-1))
    cv_f1 = np.mean(cross_val_score(clf, X_train, y_train, cv=cv_folds, scoring='f1', n_jobs=-1))
    
    overfit = "Yes" if (accuracy_gap > 0.05 or roc_auc >= 0.999) else "No"
    
    results[name] = {
        "train_accuracy": float(train_acc),
        "test_accuracy": float(test_acc),
        "accuracy_gap": float(accuracy_gap),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "roc_auc": float(roc_auc),
        "confusion_matrix": cm,
        "cv_roc_auc": float(cv_roc_auc),
        "cv_f1": float(cv_f1),
        "is_overfitted": overfit
    }

## 5. View Comparison Summary

In [ ]:
res_df = pd.DataFrame(results).T
print(res_df[['train_accuracy', 'test_accuracy', 'accuracy_gap', 'roc_auc', 'f1_score', 'is_overfitted']])

## 6. Overfitting Analysis & Best Model Selection

Rules for best model:
1. Discard/warn models with accuracy_gap > 0.05 (5%).
2. Choose model with highest test ROC-AUC, then F1, then recall.

In [ ]:
best_model_name = None
best_score = -1
best_f1 = -1
best_recall = -1

overfit_report = {}

for name, m in results.items():
    gap = m['accuracy_gap']
    is_leak = m['roc_auc'] >= 0.999
    
    overfit_report[name] = {
        "accuracy_gap": gap,
        "is_overfitted": "Yes" if gap > 0.05 else "No",
        "potential_data_leakage": "Yes" if is_leak else "No"
    }
    
    if gap <= 0.05 and not is_leak:
        # Candidate model
        if m['roc_auc'] > best_score:
            best_score = m['roc_auc']
            best_f1 = m['f1_score']
            best_recall = m['recall']
            best_model_name = name
        elif abs(m['roc_auc'] - best_score) < 1e-5:
            if m['f1_score'] > best_f1:
                best_f1 = m['f1_score']
                best_recall = m['recall']
                best_model_name = name
            elif abs(m['f1_score'] - best_f1) < 1e-5:
                if m['recall'] > best_recall:
                    best_recall = m['recall']
                    best_model_name = name

if best_model_name is None:
    # Fallback to absolute best ROC-AUC if all are slightly overfitted
    print("Warning: All models exceeded 5% overfitting limit. Selecting best fallback model.")
    best_model_name = max(results.keys(), key=lambda k: (results[k]['roc_auc'], results[k]['f1_score']))

print(f"Selected best model: {best_model_name}")

## 7. Save Reports and Charts

In [ ]:
reports_dir = "../reports"
os.makedirs(reports_dir, exist_ok=True)

# 1. Save Results CSV and JSON
res_df.to_csv(f"{reports_dir}/model_comparison_results.csv", index=True)
with open(f"{reports_dir}/model_comparison_results.json", "w") as f:
    json.dump(results, f, indent=2)

# 2. Save best model txt
with open(f"{reports_dir}/best_model.txt", "w") as f:
    f.write(f"Best Model: {best_model_name}\n")
    for k, v in results[best_model_name].items():
        f.write(f"{k}: {v}\n")

# 3. Save overfitting report
with open(f"{reports_dir}/model_overfitting_report.json", "w") as f:
    json.dump(overfit_report, f, indent=2)

# 4. Save markdown summary
summary_md = f"""# Model Comparison Summary

## Best Model Selected: {best_model_name}

| Model | Test Accuracy | Train-Test Gap | Test ROC-AUC | Test F1-Score | Overfit? |
|---|---|---|---|---|---|
"""
for name, m in results.items():
    summary_md += f"| {name} | {m['test_accuracy']:.4f} | {m['accuracy_gap']:.4f} | {m['roc_auc']:.4f} | {m['f1_score']:.4f} | {m['is_overfitted']} |\n"

with open(f"{reports_dir}/model_comparison_summary.md", "w") as f:
    f.write(summary_md)

print("All reports saved successfully.")

## 8. Generate and Save Visualizations

In [ ]:
# 1. Model comparison metrics chart
plt.figure(figsize=(10, 6))
metrics_to_plot = ['test_accuracy', 'roc_auc', 'f1_score']
plot_df = res_df[metrics_to_plot].reset_index().rename(columns={'index': 'Model'})
melt_df = pd.melt(plot_df, id_vars=['Model'], value_vars=metrics_to_plot, var_name='Metric', value_name='Score')
sns.barplot(data=melt_df, x='Model', y='Score', hue='Metric', palette='viridis')
plt.title('Model Performance Comparison')
plt.ylim(0, 1.05)
plt.tight_layout()
plt.savefig(f"{reports_dir}/model_comparison_metrics.png")
plt.close()

# 2. Confusion matrix of the best model
best_clf = models[best_model_name]
best_cm = np.array(results[best_model_name]['confusion_matrix'])
plt.figure(figsize=(6, 5))
sns.heatmap(best_cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Stable', 'Churn'], yticklabels=['Stable', 'Churn'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(f"{reports_dir}/confusion_matrix_best_model.png")
plt.close()

# 3. ROC Curve comparison
plt.figure(figsize=(10, 8))
for name, clf in models.items():
    probs = clf.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC={results[name]['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.tight_layout()
plt.savefig(f"{reports_dir}/roc_curve_comparison.png")
plt.close()

# 4. Feature importance chart of the best model
plt.figure(figsize=(10, 6))
if hasattr(best_clf, "feature_importances_"):
    importances = best_clf.feature_importances_
elif hasattr(best_clf, "coef_"):
    importances = np.abs(best_clf.coef_[0])
else:
    importances = np.zeros(X_train.shape[1])

feat_importances = pd.Series(importances, index=X_train.columns).sort_values(ascending=True)
feat_importances.tail(15).plot(kind='barh', color='teal')
plt.title(f'Feature Importance - {best_model_name}')
plt.tight_layout()
plt.savefig(f"{reports_dir}/feature_importance_best_model.png")
plt.close()

print("All charts saved successfully.")